In [1]:
import gc

In [2]:
from pathlib import Path

import numpy as np
import xarray as xr

In [3]:
from data import (
    _load_pressure_dataset,
    _load_single_dataset,
    ds_pressure,
    ds_single,
)

In [4]:
from dataclasses import dataclass


@dataclass
class CompressionRatios:
    percentile: list[float]
    absolute: list[float]
    relative: list[float]
    quadratic: list[float]

In [5]:
from compression_recommendations import Recommendations
from compression_recommendations.requirements.error_bounds.max import (
    MaxPointwiseAbsoluteErrorBoundRequirement,
    MaxPointwiseQuadraticErrorBoundRequirement,
    MaxPointwiseRelativeErrorBoundRequirement,
)

In [6]:
with Path("recommendations", "ensemble-spread.yaml").open("r") as f:
    ensemble_recommendations = Recommendations.load(f)

In [7]:
from compression_safeguards import Safeguards
from compression_safeguards.utils.bindings import Bindings
from numcodecs_combinators.stack import CodecStack
from numcodecs_shuffle import TypedByteShuffleCodec
from numcodecs_wasm_zstd import Zstd

In [8]:
# byte shuffle + default Zstandard
lossless = CodecStack(
    TypedByteShuffleCodec(),
    Zstd(level=3),
)

In [9]:
def compute_absolute_error_compression_ratio(mean: np.ndarray, eb_abs: float) -> float:
    # absolute error bound: |x' - x| <= eb_abs
    sg = Safeguards(
        safeguards=[{"kind": "eb", "type": "abs", "eb": eb_abs, "equal_nan": True}]
    )

    quantized = sg.compute_correction(data=mean, approximation=np.zeros_like(mean))
    compressed = lossless.encode(quantized)

    return mean.nbytes / compressed.nbytes


def compute_relative_error_compression_ratio(mean: np.ndarray, eb_rel: float) -> float:
    # relative error bound: |x' - x| <= |x| * eb_rel
    sg = Safeguards(
        safeguards=[{"kind": "eb", "type": "rel", "eb": eb_rel, "equal_nan": True}]
    )

    quantized = sg.compute_correction(data=mean, approximation=np.zeros_like(mean))
    compressed = lossless.encode(quantized)

    return mean.nbytes / compressed.nbytes


def compute_quadratic_error_compression_ratio(
    mean: np.ndarray, eb_qua: float, amin: float, amax: float
) -> float:
    # quadratic error bound: |x' - x| <= (1 - x^2) * eb_qua where -1 <= x <= 1
    eb_abs = (
        np.fmax(
            0.0, eb_qua - eb_qua * np.square(-1.0 + 2.0 * (mean - amin) / (amax - amin))
        )
        if amax > amin
        else 0.0
    )

    sg = Safeguards(
        safeguards=[{"kind": "eb", "type": "abs", "eb": "eb_abs", "equal_nan": True}]
    )

    quantized = sg.compute_correction(
        data=mean, approximation=np.zeros_like(mean), late_bound=Bindings(eb_abs=eb_abs)
    )
    compressed = lossless.encode(quantized)

    return mean.nbytes / compressed.nbytes


def compute_compression_ratios(
    mean: xr.DataArray, percentile: list[float], extra_markers: dict
) -> CompressionRatios:
    da = mean
    mean = mean.copy(deep=True).values

    absolute = []
    for p in percentile:
        (requirement,) = ensemble_recommendations.search(
            markers={
                "cf-short-name": da.name,
                "tags": f"absolute,{p * 100}%",
                **extra_markers,
            }
        )
        assert isinstance(requirement, MaxPointwiseAbsoluteErrorBoundRequirement), repr(
            requirement
        )
        eb_abs = requirement.value
        if not np.isfinite(eb_abs):
            absolute.append(np.nan)
        else:
            absolute.append(compute_absolute_error_compression_ratio(mean, eb_abs))
    relative = []
    for p in percentile:
        (requirement,) = ensemble_recommendations.search(
            markers={
                "cf-short-name": da.name,
                "tags": f"relative,{p * 100}%",
                **extra_markers,
            }
        )
        assert isinstance(requirement, MaxPointwiseRelativeErrorBoundRequirement), repr(
            requirement
        )
        eb_rel = requirement.value
        if not np.isfinite(eb_rel):
            relative.append(np.nan)
        else:
            relative.append(compute_relative_error_compression_ratio(mean, eb_rel))
    quadratic = []
    for p in percentile:
        (requirement,) = ensemble_recommendations.search(
            markers={
                "cf-short-name": da.name,
                "tags": f"quadratic,{p * 100}%",
                **extra_markers,
            }
        )
        assert isinstance(requirement, MaxPointwiseQuadraticErrorBoundRequirement), (
            repr(requirement)
        )
        eb_qua, amin, amax = requirement.value, requirement.minimum, requirement.maximum
        if not np.isfinite(eb_qua):
            quadratic.append(np.nan)
        else:
            quadratic.append(
                compute_quadratic_error_compression_ratio(mean, eb_qua, amin, amax)
            )

    return CompressionRatios(
        percentile=percentile,
        absolute=absolute,
        relative=relative,
        quadratic=quadratic,
    )

In [10]:
from tqdm import tqdm

In [11]:
with open("compression-ratio-single.csv", "w") as f:
    print("name,level,percentile,kind,cr", flush=True, file=f)
    for i, key in tqdm(enumerate(sorted(ds_single.mean)), position=0):
        if (i > 0) and (i % 50 == 0):
            del ds_single
            gc.collect()
            ds_single = _load_single_dataset()
        crs = compute_compression_ratios(
            ds_single.mean[key], [0.0, 0.001, 0.01, 0.05], {"level-kind": "single"}
        )
        for p, absolute, relative, quadratic in zip(
            crs.percentile, crs.absolute, crs.relative, crs.quadratic
        ):
            print(f"{key},single,{p},absolute,{absolute}", flush=True, file=f)
            print(f"{key},single,{p},relative,{relative}", flush=True, file=f)
            print(f"{key},single,{p},quadratic,{quadratic}", flush=True, file=f)
del ds_single
gc.collect()
ds_single = _load_single_dataset()

226it [28:19:27, 451.18s/it]


In [12]:
with open("compression-ratio-pressure.csv", "w") as f:
    print("name,level,percentile,kind,cr", flush=True, file=f)
    for i, key in tqdm(enumerate(sorted(ds_pressure.mean)), position=0):
        if (i > 0) and (i % 8 == 0):
            del ds_pressure
            gc.collect()
            ds_pressure = _load_pressure_dataset()
        for pressure in sorted(ds_pressure.mean[key].pressure_level.values):
            crs = compute_compression_ratios(
                ds_pressure.mean[key].sel(pressure_level=pressure),
                [0.0, 0.001, 0.01, 0.05],
                {"level-kind": "pressure", "level-value": pressure},
            )
            for p, absolute, relative, quadratic in zip(
                crs.percentile, crs.absolute, crs.relative, crs.quadratic
            ):
                print(
                    f"{key},{pressure}hPa,{p},absolute,{absolute}", flush=True, file=f
                )
                print(
                    f"{key},{pressure}hPa,{p},relative,{relative}", flush=True, file=f
                )
                print(
                    f"{key},{pressure}hPa,{p},quadratic,{quadratic}", flush=True, file=f
                )
del ds_pressure
gc.collect()
ds_pressure = _load_pressure_dataset()

16it [7:19:53, 1649.59s/it]
